### NLU Assignment 1

Name: Muhammad Fahad Waqar<br>
Student No: st125981

### Task 1

In [33]:
# Importing libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import time
from collections import Counter
import pickle
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine

In [34]:
# Importing NLTK for corpus
import nltk
nltk.download('reuters')
nltk.download('punkt')
from nltk.corpus import reuters

[nltk_data] Downloading package reuters to
[nltk_data]     C:\Users\mfaha\AppData\Roaming\nltk_data...
[nltk_data]   Package reuters is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mfaha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [35]:
# Preparing the dataset
categories = ['acq', 'crude', 'earn', 'grain', 'trade']

corpus_raw = []
for category in categories:
    fileids = reuters.fileids(category)[:100]  # Limit files per category
    for fileid in fileids:
        words = reuters.words(fileid)
        sentence = [w.lower() for w in words if w.isalpha()]
        if len(sentence) > 5:
            corpus_raw.append(sentence)

print(f"Loaded {len(corpus_raw)} sentences")

# Building vocabulary
flatten = lambda l: [item for sublist in l for item in sublist]
all_words = flatten(corpus_raw)
word_counts = Counter(all_words)

# Keep words that appear at least 5 times
vocab = [word for word, count in word_counts.items() if count >= 5]
vocab.append('<UNK>')

word2index = {w: i for i, w in enumerate(vocab)}
index2word = {i: w for w, i in word2index.items()}
voc_size = len(vocab)

print(f"Vocabulary size: {voc_size}")

Loaded 495 sentences
Vocabulary size: 1996


In [36]:
# Preparing training data
def create_skipgrams(corpus, word2index, window_size=2):
    skip_grams = []
    for sent in corpus:
        for i in range(window_size, len(sent) - window_size):
            center = sent[i] if sent[i] in word2index else '<UNK>'
            center_idx = word2index[center]

            for j in range(i - window_size, i + window_size + 1):
                if j != i:
                    context = sent[j] if sent[j] in word2index else '<UNK>'
                    skip_grams.append([center_idx, word2index[context]])
    return skip_grams

In [37]:
# Model definition
class Skipgram(nn.Module):
    def __init__(self, vocab_size, emb_size):
        super().__init__()
        self.embedding_v = nn.Embedding(vocab_size, emb_size)
        self.embedding_u = nn.Embedding(vocab_size, emb_size)

    def forward(self, center, target, all_vocab):
        v = self.embedding_v(center)
        u = self.embedding_u(target)
        u_all = self.embedding_u(all_vocab)

        score = u.bmm(v.transpose(1, 2)).squeeze()
        norm = u_all.bmm(v.transpose(1, 2)).squeeze()
        loss = -torch.mean(torch.log(torch.exp(score) / torch.sum(torch.exp(norm), 1)))
        return loss


class SkipgramNegSampling(nn.Module):
    def __init__(self, vocab_size, emb_size):
        super().__init__()
        self.embedding_v = nn.Embedding(vocab_size, emb_size)
        self.embedding_u = nn.Embedding(vocab_size, emb_size)
        self.logsigmoid = nn.LogSigmoid()

    def forward(self, center, target, negatives):
        v = self.embedding_v(center)
        u = self.embedding_u(target)
        neg_u = -self.embedding_u(negatives)

        pos_score = u.bmm(v.transpose(1, 2)).squeeze()
        neg_score = neg_u.bmm(v.transpose(1, 2))

        loss = self.logsigmoid(pos_score) + torch.sum(self.logsigmoid(neg_score), 1)
        return -torch.mean(loss)


class GloVe(nn.Module):
    def __init__(self, vocab_size, emb_size):
        super().__init__()
        self.v = nn.Embedding(vocab_size, emb_size)
        self.u = nn.Embedding(vocab_size, emb_size)
        self.v_bias = nn.Embedding(vocab_size, 1)
        self.u_bias = nn.Embedding(vocab_size, 1)

    def forward(self, wi, wj, xij, weight):
        vi = self.v(wi)
        uj = self.u(wj)
        bi = self.v_bias(wi).squeeze()
        bj = self.u_bias(wj).squeeze()
        loss = weight * (torch.sum(vi * uj, 1) + bi + bj - xij) ** 2
        return torch.mean(loss)

In [38]:
# Training function
def train_model(model_type, window_size=2, emb_size=100, epochs=5, batch_size=512):
    skipgrams = create_skipgrams(corpus_raw, word2index, window_size)

    if model_type == 'skipgram':
        model = Skipgram(voc_size, emb_size)
    elif model_type == 'skipgram_neg':
        model = SkipgramNegSampling(voc_size, emb_size)

    optimizer = optim.Adam(model.parameters(), lr=0.01)
    losses = []
    start = time.time()

    for epoch in range(epochs):
        total_loss = 0
        for i in range(0, len(skipgrams) - batch_size, batch_size):
            batch = skipgrams[i:i+batch_size]
            center = torch.LongTensor([[x[0]] for x in batch])
            target = torch.LongTensor([[x[1]] for x in batch])

            optimizer.zero_grad()

            if model_type == 'skipgram':
                all_vocab = torch.LongTensor(range(voc_size)).expand(batch_size, voc_size)
                loss = model(center, target, all_vocab)
            else:
                neg = torch.randint(0, voc_size, (batch_size, 10))
                loss = model(center, target, neg)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        losses.append(total_loss / len(skipgrams))
        print(f"{model_type} Epoch {epoch+1}, Loss: {losses[-1]:.4f}")

    return model, time.time() - start, losses

In [39]:
results = {}
for model_name in ['skipgram', 'skipgram_neg']:
    model, t, l = train_model(model_name)
    results[f"{model_name}_w2"] = {
        "model": model,
        "train_time": t,
        "losses": l
    }

skipgram Epoch 1, Loss: 0.0256
skipgram Epoch 2, Loss: 0.0134
skipgram Epoch 2, Loss: 0.0134
skipgram Epoch 3, Loss: 0.0114
skipgram Epoch 3, Loss: 0.0114
skipgram Epoch 4, Loss: 0.0108
skipgram Epoch 4, Loss: 0.0108
skipgram Epoch 5, Loss: 0.0106
skipgram Epoch 5, Loss: 0.0106
skipgram_neg Epoch 1, Loss: 0.0276
skipgram_neg Epoch 1, Loss: 0.0276
skipgram_neg Epoch 2, Loss: 0.0070
skipgram_neg Epoch 2, Loss: 0.0070
skipgram_neg Epoch 3, Loss: 0.0049
skipgram_neg Epoch 3, Loss: 0.0049
skipgram_neg Epoch 4, Loss: 0.0041
skipgram_neg Epoch 4, Loss: 0.0041
skipgram_neg Epoch 5, Loss: 0.0037
skipgram_neg Epoch 5, Loss: 0.0037


In [40]:
# Training GloVe on the same corpus
def build_cooccurrence(corpus, word2index, window_size=2):
    counts = Counter()
    for sent in corpus:
        for i, w in enumerate(sent):
            if w not in word2index:
                continue
            wi = word2index[w]
            start = max(0, i - window_size)
            end = min(len(sent), i + window_size + 1)
            for j in range(start, end):
                if i == j:
                    continue
                wj = sent[j]
                if wj not in word2index:
                    continue
                counts[(wi, word2index[wj])] += 1
    return counts

def train_glove(window_size=2, emb_size=100, epochs=5, batch_size=1024, x_max=100, alpha=0.75):
    co_counts = build_cooccurrence(corpus_raw, word2index, window_size)
    pairs = list(co_counts.items())
    model = GloVe(voc_size, emb_size)
    optimizer = optim.Adagrad(model.parameters(), lr=0.05)
    losses = []
    start = time.time()
    for epoch in range(epochs):
        np.random.shuffle(pairs)
        total_loss = 0
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i+batch_size]
            wi = torch.LongTensor([p[0][0] for p in batch])
            wj = torch.LongTensor([p[0][1] for p in batch])
            xij_raw = torch.FloatTensor([p[1] for p in batch])
            weight = torch.FloatTensor([(x.item() / x_max) ** alpha if x < x_max else 1.0 for x in xij_raw])
            xij = torch.log(xij_raw)
            optimizer.zero_grad()
            loss = model(wi, wj, xij, weight)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        losses.append(total_loss / len(pairs))
        print(f"glove Epoch {epoch+1}, Loss: {losses[-1]:.4f}")
    return model, time.time() - start, losses

glove_model, glove_time, glove_losses = train_glove(window_size=2, emb_size=100, epochs=5, batch_size=1024)
results['glove'] = {
    "model": glove_model,
    "train_time": glove_time,
    "losses": glove_losses
}

glove Epoch 1, Loss: 0.0048
glove Epoch 2, Loss: 0.0027
glove Epoch 2, Loss: 0.0027
glove Epoch 3, Loss: 0.0020
glove Epoch 3, Loss: 0.0020
glove Epoch 4, Loss: 0.0017
glove Epoch 4, Loss: 0.0017
glove Epoch 5, Loss: 0.0014
glove Epoch 5, Loss: 0.0014


In [41]:
# Loading pre-trained GloVe from Gensim for comparison
try:
    import gensim.downloader as api
    print("Loading pre-trained GloVe model from Gensim...")
    pretrained_glove = api.load('glove-wiki-gigaword-100')
    print("Pre-trained GloVe loaded successfully")
    
    # Store in results for comparison
    results['glove_gensim'] = {
        'model': pretrained_glove,
        'train_time': 0,  # Pre-trained, no training time
        'losses': [],
        'is_pretrained': True
    }
except Exception as e:
    print(f"Note: Could not load pre-trained GloVe: {e}")
    pretrained_glove = None


print("Model Comparison Table")

# Prepare table data
table_data = []

# Helper function to get analogy accuracy
def get_embedding_for_eval(model, word, is_pretrained=False):
    if is_pretrained:
        try:
            return model[word]
        except:
            return None
    else:
        if word not in word2index:
            return None
        idx = word2index[word]
        with torch.no_grad():
            if isinstance(model, GloVe):
                vec = (model.v.weight[idx] + model.u.weight[idx]) / 2
                return vec.cpu().numpy()
            else:
                idx_t = torch.LongTensor([idx])
                v = model.embedding_v(idx_t)
                u = model.embedding_u(idx_t)
                return ((v + u) / 2).detach().numpy()[0]

def quick_analogy_accuracy(model, filepath, category, is_pretrained=False, max_samples=50):
    correct, total = 0, 0
    current = None
    with open(filepath) as f:
        for line in f:
            if total >= max_samples:
                break
            if line.startswith(":"):
                current = line.strip()
                continue
            if current is None or category not in current:
                continue
            words = line.lower().split()
            if len(words) != 4:
                continue
            a, b, c, d = words
            
            # Get embeddings
            vec_a = get_embedding_for_eval(model, a, is_pretrained)
            vec_b = get_embedding_for_eval(model, b, is_pretrained)
            vec_c = get_embedding_for_eval(model, c, is_pretrained)
            
            if vec_a is None or vec_b is None or vec_c is None:
                continue
                
            target = vec_b - vec_a + vec_c
            
            # Find best match
            best, best_sim = None, -1
            
            if is_pretrained:
                # For pretrained, search in its vocabulary
                try:
                    similar = model.most_similar(positive=[b, c], negative=[a], topn=1)
                    if similar and similar[0][0] != a and similar[0][0] != b and similar[0][0] != c:
                        best = similar[0][0]
                except:
                    pass
            else:
                # For our models, search in our vocab
                search_vocab = vocab[:100]  # Limit search for speed
                for w in search_vocab:
                    if w in [a, b, c]:
                        continue
                    vec_w = get_embedding_for_eval(model, w, is_pretrained)
                    if vec_w is None:
                        continue
                    sim = 1 - cosine(target, vec_w)
                    if sim > best_sim:
                        best, best_sim = w, sim
            
            if best == d:
                correct += 1
            total += 1
    
    return (correct / total * 100) if total > 0 else 0

# Calculate metrics for each model
model_configs = [
    ('skipgram_w2', 'Skipgram', 2, False),
    ('skipgram_neg_w2', 'Skipgram (NEG)', 2, False),
    ('glove', 'GloVe', 2, False),
]

if pretrained_glove:
    model_configs.append(('glove_gensim', 'GloVe (Gensim)', '-', True))

for model_key, model_name, window_size, is_pretrained in model_configs:
    if model_key not in results:
        continue
    
    model_info = results[model_key]
    model = model_info['model']
    
    # Basic metrics
    row = {
        'Model': model_name,
        'Window Size': window_size,
        'Training Loss': '-' if is_pretrained else f"{model_info['losses'][-1]:.4f}",
        'Training Time': '-' if is_pretrained else f"{model_info['train_time']:.2f}s",
    }
    
    # Calculate accuracies
    if not is_pretrained:
        syntactic_acc = quick_analogy_accuracy(model, 'word-test.v1.txt', 'past-tense', is_pretrained, max_samples=30)
        semantic_acc = quick_analogy_accuracy(model, 'word-test.v1.txt', 'capital-common-countries', is_pretrained, max_samples=30)
        row['Syntactic Accuracy'] = f"{syntactic_acc:.1f}%"
        row['Semantic Accuracy'] = f"{semantic_acc:.1f}%"
    else:
        # For pretrained, use different evaluation
        row['Syntactic Accuracy'] = 'N/A (diff vocab)'
        row['Semantic Accuracy'] = 'N/A (diff vocab)'
    
    table_data.append(row)

# Print formatted table
print("\n")
print(f"{'Model':<20} | {'Window Size':<12} | {'Training Loss':<14} | {'Training Time':<14} | {'Syntactic Accuracy':<19} | {'Semantic Accuracy':<19}")
print("-" * 130)

for row in table_data:
    print(f"{row['Model']:<20} | {str(row['Window Size']):<12} | {row['Training Loss']:<14} | {row['Training Time']:<14} | {row['Syntactic Accuracy']:<19} | {row['Semantic Accuracy']:<19}")

def get_embedding(model, word):
    idx = word2index[word]
    with torch.no_grad():
        if isinstance(model, GloVe):
            vec = (model.v.weight[idx] + model.u.weight[idx]) / 2
            return vec.cpu().numpy()
        else:
            idx_t = torch.LongTensor([idx])
            v = model.embedding_v(idx_t)
            u = model.embedding_u(idx_t)
            return ((v + u) / 2).detach().numpy()[0]

def analogy_accuracy(model, filepath, category):
    correct, total = 0, 0
    current = None
    with open(filepath) as f:
        for line in f:
            if line.startswith(":"):
                current = line.strip()
                continue
            if current is None or category not in current:
                continue
            a, b, c, d = line.lower().split()
            if any(w not in word2index for w in [a, b, c, d]):
                continue
            target = get_embedding(model, b) - get_embedding(model, a) + get_embedding(model, c)
            best, best_sim = None, -1
            for w in vocab:
                if w in [a, b, c]:
                    continue
                sim = 1 - cosine(target, get_embedding(model, w))
                if sim > best_sim:
                    best, best_sim = w, sim
            correct += (best == d)
            total += 1
    return correct / total if total else 0

analogy_models = {
    'skipgram_neg_w2': 'Skipgram Negative Sampling',
    'glove': 'GloVe'
}

for key, label in analogy_models.items():
    mdl = results.get(key, {}).get('model')
    if mdl is None:
        continue
    print(f"{label} (Past Tense):", analogy_accuracy(mdl, 'word-test.v1.txt', 'past-tense'))
    print(f"{label} (Countries):", analogy_accuracy(mdl, 'word-test.v1.txt', 'capital-common-countries'))

Loading pre-trained GloVe model from Gensim...
[==================================================] 100.0% 128.1/128.1MB downloaded

Pre-trained GloVe loaded successfully
Model Comparison Table
Pre-trained GloVe loaded successfully
Model Comparison Table


Model                | Window Size  | Training Loss  | Training Time  | Syntactic Accuracy  | Semantic Accuracy  
----------------------------------------------------------------------------------------------------------------------------------
Skipgram             | 2            | 0.0106         | 604.13s        | 0.0%                | 0.0%               
Skipgram (NEG)       | 2            | 0.0037         | 23.80s         | 0.0%                | 0.0%               
GloVe                | 2            | 0.0014         | 9.27s          | 0.0%                | 0.0%               
GloVe (Gensim)       | -            | -              | -              | N/A (diff vocab)    | N/A (diff vocab)   


Model                | Window Size  | Tr

In [42]:
# Similarity Correlation
def similarity_correlation(model, filepath):
    df = pd.read_csv(filepath)
    human, model_sim = [], []
    for _, row in df.iterrows():
        w1, w2 = row['Word 1'].lower(), row['Word 2'].lower()
        if w1 not in word2index or w2 not in word2index:
            continue
        vec1 = get_embedding(model, w1)
        vec2 = get_embedding(model, w2)
        model_sim.append(1 - cosine(vec1, vec2))
        human.append(float(row['Human (mean)']))
    return spearmanr(model_sim, human)[0] if human else 0.0

print("\nSpearman Correlation")
for key, label in {
    'skipgram_neg_w2': 'Skipgram Negative Sampling',
    'glove': 'GloVe'
}.items():
    mdl = results.get(key, {}).get('model')
    if mdl is None:
        continue
    print(f"{label}:", similarity_correlation(mdl, 'wordsim353.csv'))


Spearman Correlation
Skipgram Negative Sampling: 0.08140800780779905
GloVe: -0.03532115643369587


### Task 3 - Model Saving & Application

In [44]:
import os

# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save vocabulary and metadata
vocab_data = {
    'word2index': word2index,
    'index2word': index2word,
    'vocab': vocab,
    'voc_size': voc_size
}

with open('models/vocabulary.pkl', 'wb') as f:
    pickle.dump(vocab_data, f)
print("Saved vocabulary to models/vocabulary.pkl")

# Save each model with embeddings
for model_name, model_info in results.items():
    model = model_info['model']
    
    # Skip pretrained models
    if model_info.get('is_pretrained', False):
        print(f"Skipping {model_name} (pretrained model, cannot save in same format)")
        continue
    
    # Extract embeddings from the model
    if isinstance(model, GloVe):
        # For GloVe: average v and u embeddings
        embeddings = (model.v.weight.detach().cpu().numpy() + 
                     model.u.weight.detach().cpu().numpy()) / 2
    else:
        # For Skipgram variants: average embedding_v and embedding_u
        embeddings = (model.embedding_v.weight.detach().cpu().numpy() + 
                     model.embedding_u.weight.detach().cpu().numpy()) / 2
    
    # Saving model data
    model_data = {
        'embeddings': embeddings,
        'model_type': model_name,
        'train_time': model_info['train_time'],
        'losses': model_info['losses'],
        'final_loss': model_info['losses'][-1],
        'vocab_size': voc_size,
        'embedding_dim': embeddings.shape[1]
    }
    
    filename = f'models/{model_name}.pkl'
    with open(filename, 'wb') as f:
        pickle.dump(model_data, f)
    print(f"Saved {model_name} to {filename} ({embeddings.shape})")

torch.save(results['skipgram_w2']['model'].state_dict(), 'models/skipgram_w2_state.pth')
torch.save(results['skipgram_neg_w2']['model'].state_dict(), 'models/skipgram_neg_w2_state.pth')
torch.save(results['glove']['model'].state_dict(), 'models/glove_state.pth')

Saved vocabulary to models/vocabulary.pkl
Saved skipgram_w2 to models/skipgram_w2.pkl ((1996, 100))
Saved skipgram_neg_w2 to models/skipgram_neg_w2.pkl ((1996, 100))
Saved glove to models/glove.pkl ((1996, 100))
Skipping glove_gensim (pretrained model, cannot save in same format)


In [45]:
def load_model_for_app(model_name='skipgram_neg_w2'):
    # Load vocabulary
    with open('models/vocabulary.pkl', 'rb') as f:
        vocab_data = pickle.load(f)
    
    # Load model
    with open(f'models/{model_name}.pkl', 'rb') as f:
        model_data = pickle.load(f)
    
    return model_data['embeddings'], vocab_data['word2index'], vocab_data['index2word']

def find_similar_words(word, embeddings, word2index, index2word, top_k=5):
    if word not in word2index:
        return f"Word '{word}' not in vocabulary"
    
    word_idx = word2index[word]
    word_vec = embeddings[word_idx]
    
    # Calculate cosine similarities
    similarities = []
    for idx in range(len(embeddings)):
        if idx == word_idx:
            continue
        sim = 1 - cosine(word_vec, embeddings[idx])
        similarities.append((index2word[idx], sim))
    
    # Sort and return top k
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

def solve_analogy_demo(a, b, c, embeddings, word2index, index2word, top_k=3):
    if any(w not in word2index for w in [a, b, c]):
        return "One or more words not in vocabulary"
    
    vec_a = embeddings[word2index[a]]
    vec_b = embeddings[word2index[b]]
    vec_c = embeddings[word2index[c]]
    
    target = vec_b - vec_a + vec_c
    
    # Find closest words (excluding a, b, c)
    similarities = []
    for idx in range(len(embeddings)):
        word = index2word[idx]
        if word in [a, b, c]:
            continue
        sim = 1 - cosine(target, embeddings[idx])
        similarities.append((word, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

# Example usage
# Load the best performing model (skipgram_neg_w2)
emb, w2i, i2w = load_model_for_app('skipgram_neg_w2')
print(f"\nLoaded model: skipgram_neg_w2")
print(f"Vocabulary size: {len(w2i)}")
print(f"Embedding dimension: {emb.shape[1]}")

print("Demo 1: Similar Words")
test_words = ['oil', 'bank', 'trade']
for word in test_words:
    if word in w2i:
        similar = find_similar_words(word, emb, w2i, i2w, top_k=5)
        print(f"\nWords similar to '{word}':")
        for w, sim in similar:
            print(f"  {w:<15} (similarity: {sim:.4f})")

# Demo 2: Solve analogies
print("Demo 2: Word Analogies")
analogies = [
    ('man', 'woman', 'king'),
    ('paris', 'france', 'london'),
    ('buy', 'bought', 'sell')
]
for a, b, c in analogies:
    result = solve_analogy_demo(a, b, c, emb, w2i, i2w, top_k=3)
    if isinstance(result, str):
        print(f"\n{a}:{b} :: {c}:? → {result}")
    else:
        print(f"\n{a}:{b} :: {c}:?")
        for w, sim in result:
            print(f"  {w:<15} (similarity: {sim:.4f})")


Loaded model: skipgram_neg_w2
Vocabulary size: 1996
Embedding dimension: 100
Demo 1: Similar Words

Words similar to 'oil':
  crude           (similarity: 0.4636)
  natural         (similarity: 0.3803)
  postings        (similarity: 0.3720)
  deregulation    (similarity: 0.3471)
  charges         (similarity: 0.3328)

Words similar to 'bank':
  central         (similarity: 0.3729)
  sumitomo        (similarity: 0.3605)
  research        (similarity: 0.3076)
  savings         (similarity: 0.2879)
  national        (similarity: 0.2847)

Words similar to 'trade':
  s               (similarity: 0.5091)
  and             (similarity: 0.3958)
  u               (similarity: 0.3850)
  the             (similarity: 0.3741)
  surplus         (similarity: 0.3645)
Demo 2: Word Analogies

man:woman :: king:? → One or more words not in vocabulary

paris:france :: london:?
  britain         (similarity: 0.3880)
  decision        (similarity: 0.3177)
  have            (similarity: 0.3126)

buy:bought 